# Algorithm how to compute "all" slices of a polytope

This algorithm follows from observations made in [The Best Ways to Slice a Polytope](https://arxiv.org/pdf/2304.14239) by Marie-Charlotte Brandenburg, Jesús De Loera and Chiara Meroni.

At a [Dive into Research at Ruhr Universität Bochum](https://www.combinatorial-synergies.de/activities/2025-08_DiR-Bochum/) in the summer of 2025 our group (Anna Birkemeyer, Marie-Charlotte Brandenburg, Niklas Prün and myself) implemented the algorithm to compute all central, or affine slices of a polytope. We also studied combinatorial properties of polytopes with respect to combinatorial properties of slices, see our preprint for more: [Combinatorial slicing problems of polytopes:
How (not) to reconstruct a polytope from its slices](https://arxiv.org/pdf/2609.16195)

In the following I will explain why and how the algorithm works, and go through the code step-by-step.

## What the algorithm does, and why it works

**Observation 1:** The combinatorial type of a slice $P \cap H$ is uniquely determined by the faces of $P$ that $H$ intersects in the relative interior. To see this; the face lattice of $P \cap H$ is the induced subposet of the face lattice of $P$, on the faces whose relative interior is intersected by $H$.

In light of this, we call two slices $P$-equivalent if they intersect the same set of faces of $P$ in the relative interior.

**The algorithm we present below computes one slice per $P$-equivalence class of slices.**

**Observation 2:** The faces of $P$ that a hyperplane $H$ intersects is uniquely determined by the partition of the vertices by $H$ into $V^+ := \text{verts}(P) \cap H^+$, $V^0 := \text{verts}(P) \cap H^-$ and $V := \text{verts}(P) \cap H$. Thus a $P$-equivalence class of slices corresponds to a partition of $\text{verts}(P)$ into the sets $V^+, V^-, V^0$.

**We present a hyperplane arrangement that partitions the parameter space of affine hyperplanes into finitely many cells, each corresponding to a $P$-equivalence class of slices.**

- We identify a hyperplane $H_u(\beta):= \{ x \colon \langle u,x\rangle + \beta =0 \}$ with its parameters $(u,\beta)$.
- The slice $P \cap H_u(\beta)$ partitions the vertices of $P$ via $v \in  V^{\text{sgn}(\langle u,v\rangle + \beta)}$. That is, if $\langle u,v\rangle+\beta >0$, then $v \in V^+$, etc.


Consider the hyperplane arrangement $\mathcal H(P \times \{1\}) := \{(v,1)^\perp \colon v \in \text{verts}(P)\}.$ We call its maximally relatively open cones *cells*. They are separated by the hyperplanes $(v,1)^\perp$. Every cell $C$ corresponds to a sign vector $\sigma(C) \in \{+,-,0\}^V$, indexed by vertices of $P$, such that the sign $\sigma_v(C)$ is the side of the hyperplane $H=(v,1)^\perp$ that this cell is on.

**Claim:** A cell $C$ and it's opposite $-C$ correspond to a $P$-equivalence class of slices.

To see this, if $\pm (u,\beta) \in C$, then $\sigma_v(C) = \langle (u,\beta),(v,1) \rangle  = \langle u,v\rangle + \beta$. Since $\sigma_v(C)$ is independent of the choice of $(u,\beta) \in C$, every $(u,\beta) \in C$ induces the same partition of vertices of $P$ into
$$  V^+ = \{v \colon \sigma(C)_v = +\}, \quad V^- = \{v \colon \sigma(C)_v = -\}, \quad V^0 = \{v \colon \sigma(C)_v = 0\}.$$

**Algorithm to compute one slice per $P$-equivalence class of slices of $P$.**
  1. Compute $\mathcal H(P \times \{1\}) = \{(v,1)^\perp \colon v \in \text{verts}(P)\}$.
  2. For each cell $C$ of $\mathcal H (P \times \{1\})$, sample a vector $(u,\beta)$ from $C$.
  3. Compute the slice $S = P \cap H_u(\beta)$.

## Example: 3-cube

1. Compute $\mathcal H (P \times \{1\})$.

In [1]:
P = polytopes.cube()

In [14]:
normals = [v+[1] for v in P.vertices_list()]
hyperplane_data = [[n, 0] for n in normals] #0 for central hyperplane
H = HyperplaneArrangements(QQ, tuple(['x'+str(i) for i in range(1, P.ambient_dim()+2)])) 
HP1 = H(hyperplane_data)

for hyp in HP1:
    print(hyp)

Hyperplane -x1 - x2 - x3 + x4 + 0
Hyperplane -x1 - x2 + x3 + x4 + 0
Hyperplane -x1 + x2 - x3 + x4 + 0
Hyperplane -x1 + x2 + x3 + x4 + 0
Hyperplane x1 - x2 - x3 + x4 + 0
Hyperplane x1 - x2 + x3 + x4 + 0
Hyperplane x1 + x2 - x3 + x4 + 0
Hyperplane x1 + x2 + x3 + x4 + 0


2. For each pair of opposite cells of $\mathcal H(P \times \{1\})$, sample a vector $(u,\beta)$ will correspond to a the affine hyperplane $H_u(\beta)$.

In [4]:
def sample_normalvectors(hp_arr, all_or_generic="all"):
    """
    Input: hyperplane arrangement
    Output: one sampled vector from each cell, up to opposite cells. Obtain a vector by summing rays of the cone.
    """
    regions = hp_arr.regions()
    normalvectors = []
    
    for reg in regions: #full-dimensional regions
        rays = reg.rays() #if fails then add if len(rays)==0: continue
        n = sum([vector(list(r)) for r in rays]) #try sum(vector(QQ, list(r)) for r in rays) 
        if -n not in normalvectors:
            normalvectors.append(n)
        
            if all_or_generic == "all": #include lower-dimensional regions 
                for k in range(1, reg.dimension()): # skip empty cone
                    for face_of_region in reg.faces(k):
                        rays = face_of_region.as_polyhedron().rays()
                        n = sum([vector(list(r)) for r in rays])
                        if n not in normalvectors and -n not in normalvectors: #cause we will do many faces double
                            normalvectors.append(n)
    return normalvectors

In [65]:
normals = sample_normalvectors(HP1, all_or_generic="all")
print(f"We've sampled {len(normals)} normal vectors from distinct P-equivalent cells, i.e., not from two opposite cells.\nHere they are.")
print(normals)

We've sampled 296 normal vectors from distinct P-equivalent cells, i.e., not from two opposite cells.
Here they are.
[(0, 0, 0, 6), (-1, 0, 0, 1), (0, -1, 0, 1), (0, 0, -1, 1), (0, 0, 1, 1), (0, 1, 0, 1), (1, 0, 0, 1), (-1, -1, 0, 2), (-1, 0, -1, 2), (0, -1, -1, 2), (-1, 0, 1, 2), (0, -1, 1, 2), (-1, 1, 0, 2), (0, 1, -1, 2), (0, 1, 1, 2), (1, -1, 0, 2), (1, 0, -1, 2), (1, 0, 1, 2), (1, 1, 0, 2), (-1, -1, -1, 3), (-1, -1, 1, 3), (-1, 1, -1, 3), (-1, 1, 1, 3), (1, -1, -1, 3), (1, -1, 1, 3), (1, 1, -1, 3), (1, 1, 1, 3), (-2, -2, -2, 4), (-1, -1, -1, 1), (-1, -1, -2, 2), (-1, -2, -1, 2), (-2, -1, -1, 2), (-1, -2, -2, 3), (-2, -1, -2, 3), (-2, -2, -1, 3), (-2, -2, 2, 4), (-1, -1, 1, 1), (-2, -1, 1, 2), (-1, -1, 2, 2), (-1, -2, 1, 2), (-2, -1, 2, 3), (-2, -2, 1, 3), (-1, -2, 2, 3), (-4, -4, 0, 4), (-1, -1, 0, 0), (-2, -1, 0, 1), (-2, -2, -1, 1), (-2, -2, 1, 1), (-1, -2, 0, 1), (-3, -2, -1, 2), (-3, -2, 1, 2), (-2, -3, -1, 2), (-2, -3, 1, 2), (-2, 2, -2, 4), (-1, 1, -1, 1), (-2, 1, -1, 2), (-

3. Compute the slice $S = P \cap H_u(\beta)$.

In [66]:
slices=[]

for ubeta in normals:
    H_u_beta = Polyhedron(eqns=[[ubeta[-1]] + list(ubeta[:-1])])
    S = P.intersection(H_u_beta)
    slices.append(S)

print(f"We've computed all slice (empty and lower-dimensional slices inclusive).\nThey are all n-gons for the following n.\n0, 1 or 2 vertices correspond to \'boundary\' slices of P, i.e., the empty slice, a vertex or an edge of P.")
ngons = Set(S.n_vertices() for S in slices)
print(ngons)

We've computed all slice (empty and lower-dimensional slices inclusive).
They are all n-gons for the following n.
0, 1 or 2 vertices correspond to 'boundary' slices of P, i.e., the empty slice, a vertex or an edge of P.
{0, 1, 2, 3, 4, 5, 6}


## Let's plot some of our slices

In [67]:
G = P.plot(
    point={"color": "black", "size": 10}, line={"color": "black", "thickness": 1}, 
    polygon={"color": "lightblue", "opacity": 0.25})

#Choose a slice to plot
S = slices[45]
print(S.vertices_list())
if S.dim() != -1: #only plot slices through relint of P, or a facet 
    print(f"The slice has vertices {S.vertices_list()}.")
    G += S.plot(point={"color": "purple", "size": 20}, line={"color": "purple", "thickness": 2},
    polygon={"color": "purple", "opacity": 0.6})
    G.show(frame=False, online=True)
else:
    print(f"The slice is empty.")


[[1, -1, 1], [1, -1, -1], [0, 1, 1], [0, 1, -1]]
The slice has vertices [[1, -1, 1], [1, -1, -1], [0, 1, 1], [0, 1, -1]].


Graphics3d Object